# CART Analysis Pipeline

**What you need to do:** fill in the folder where your lifs are saved and check that the channel order matches the one you see in ImageJ (channel 0 is the first channel!). The code has 5 stages:

1. **Discover jobs** — find all .lif files in your folder and identify all the iamges contained within them. Writes `manifest.csv` so you can check that all the jobs you expect have been identified.
2. **Auto-detect masks** — segment chip, tumour, vasculature, B cells and CART cells. A .tif file is written for each image, with the channels:

| Ch | Contents | Editable? |
|---|------------|-----------|
| 0 | BF raw (uint8) | NA |
| 1 | B cells raw (uint8) | NA |
| 2 | Vasculature raw (uint8) | NA |
| 3 | CART cells raw (uint8) | NA |
| 4 | Chip center mask | **Yes** |
| 5 | Tumour mask | **Yes** |
| 6 | Left device mask | No (derived from chip mask) |
| 7 | Right device mask | No (derived from chip mask) |
| 8 | Vasculature mask | No (auto only) |
| 9 | B cells mask | No (auto only) |
| 10 | CART cells mask | No (auto only) |


3. **Curate labels** — this step lets you go through each segmentation and curate the identified tumour or chip region. You can erase or paint in extra region so that the segmentation perfectly matches the tumour/chip.
4. **Analyse** — count cells per region, compute distances from the tumour edge, and compute colocalisation metrics; write `all_counts.csv`, `all_distances.csv`, `all_colocalisation.csv`.
5. **Figures** — save a PNG of each image showing the overlaid segmentation.

In [1]:
from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd().parent))
from cart_id import discover_jobs, auto_detect, label_curation, analyse, figures

# ── Pipeline parameters (edit as needed) ─────────────────────────────────────
SOURCE_DIR = Path(r"Z:\Bel\Marina_Side_Projects\CART analysis\To_Do\Inputs")  
OUTPUT_DIR = SOURCE_DIR.parent / "Outputs"
# ─────────────────────────────────────────────────────────────────────────────

MANIFEST_PATH = OUTPUT_DIR / "manifest.csv"
CH_BCELLS = 0
CH_VASC = 1
CH_BF = 2
# CART cells = last channel

## Stage 1: Discover jobs

In [2]:
discover_jobs.build_and_write_manifest(
    source_dir=SOURCE_DIR,
    output_dir=OUTPUT_DIR,
    manifest_path=MANIFEST_PATH,
)

Manifest written: Z:\Bel\Marina_Side_Projects\CART analysis\To_Do\Outputs\manifest.csv
  16/16 images to process


WindowsPath('Z:/Bel/Marina_Side_Projects/CART analysis/To_Do/Outputs/manifest.csv')

## Stage 2: Auto-detect masks

For each processable image, segments the chip device, tumour, vasculature, B cells, and CART cells, then writes an 11-channel `stage2/<name>.tif` with the pixel size embedded in the metadata.

If you have a powerful computer, increase `n_workers`>1 to run multiple jobs simultaneously.

In [3]:
auto_detect.run(MANIFEST_PATH, ch_bf=CH_BF, ch_bcells=CH_BCELLS, ch_vasc=CH_VASC, n_workers=2)

Stage 2: processing 16/16 images (n_workers=2)


c:\Users\taylorhearn\AppData\Local\miniconda3\envs\clean_vascumap\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


  [OK  ] 2026.05.14_FL37_UTD_device1_img0
  [OK  ] 2026.05.14_FL37_ARi_device1_img1
  [OK  ] 2026.05.14_FL37_UTD_device2_img2
  [OK  ] 2026.05.14_FL37_ARi_device2_img3
  [OK  ] 2026.05.14_FL37_UTD_device3_img4
  [OK  ] 2026.05.14_FL37_ARi_device3_img5
  [OK  ] 2026.05.14_FL37_ARi_device4_img6
  [OK  ] 2026.05.14_FL37_UTD_device4_img7
  [OK  ] 2026.05.14_FL41_UTD_device1_img0
  [OK  ] 2026.05.14_FL41_ARi_device1_img1
  [OK  ] 2026.05.14_FL41_UTD_device2_img2
  [OK  ] 2026.05.14_FL41_ARi_device2_img3
  [OK  ] 2026.05.14_FL41_UTD_device3_img4
  [OK  ] 2026.05.14_FL41_ARi_device3_img5
  [OK  ] 2026.05.14_FL41_UTD_device4_img6
  [OK  ] 2026.05.14_FL41_ARi_device4_img7
Stage 2 done: 16/16 images processed.


## Stage 3: Curate labels *(optional but desirable!)*

Opens an interactive viewer showing the images and their segmentations.

napari viewer. For each image the viewer prefers `stage3/<name>.tif` (a previous curation session) and falls back to `stage2/<name>.tif` (auto). Two label layers are editable:

- **Chip center** (cyan) — the interior channel region
- **Tumour** (red) — the tumour mass

The B-cell, vasculature, and CART raw channels are togglable reference layers (hidden by default — toggle visibility in the layer list). Navigate with **Previous** / **Next**. Edited images are written to `stage3/<name>.tif` immediately on navigation, so progress survives kernel restarts. Click **Done** when finished.

Skip this cell if the auto-detected masks are good enough.

In [2]:
label_curation.main([str(MANIFEST_PATH)])

Opening curation viewer for 16 job(s)...


c:\Users\taylorhearn\AppData\Local\miniconda3\envs\clean_vascumap\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
c:\Users\taylorhearn\git_repos\image_quantification\CART_Analysis\cart_id\label_curation.py:220: FutureWarning: Public access to Window.qt_viewer is deprecated and will be removed in
v0.7.0. It is considered an "implementation detail" of the napari
application, not part of the napari viewer model. If your use case
requires access to qt_viewer, please open an issue to discuss.
  (getattr(self.viewer.window, "qt_viewer", None), "destroyed"),


0

  saved stage3/2026.05.14_FL37_UTD_device1_img0.tif
  saved stage3/2026.05.14_FL37_ARi_device1_img1.tif
  saved stage3/2026.05.14_FL37_UTD_device2_img2.tif
  saved stage3/2026.05.14_FL37_ARi_device2_img3.tif
  saved stage3/2026.05.14_FL37_UTD_device3_img4.tif
  saved stage3/2026.05.14_FL37_ARi_device3_img5.tif
  saved stage3/2026.05.14_FL37_ARi_device4_img6.tif
  saved stage3/2026.05.14_FL37_UTD_device4_img7.tif
  saved stage3/2026.05.14_FL41_UTD_device1_img0.tif
  saved stage3/2026.05.14_FL41_ARi_device1_img1.tif
  saved stage3/2026.05.14_FL41_UTD_device2_img2.tif
  saved stage3/2026.05.14_FL41_ARi_device2_img3.tif
  saved stage3/2026.05.14_FL41_UTD_device3_img4.tif
  saved stage3/2026.05.14_FL41_ARi_device3_img5.tif
  saved stage3/2026.05.14_FL41_ARi_device3_img5.tif
  saved stage3/2026.05.14_FL41_UTD_device4_img6.tif
  saved stage3/2026.05.14_FL41_ARi_device4_img7.tif
Stage 3 complete: 16/16 jobs have stage3 overrides.


## Stage 4 — Analyse

Prefers `stage3/<name>.tif` (curated) and falls back to `stage2/<name>.tif` (auto). For each image:

- **Cell counts** per region (chip, tumour, chip-not-tumour, vasculature, left/right device) for CART cells and B cells.
- **Per-cell distance** from the tumour edge (in µm) with region-membership flags.
- **Colocalisation** between CART and B cells: coarse Pearson (Gaussian blur at 5, 10, 20, 50, 100 µm) and binary proximity fraction (radii 5–100 µm).

Writes three CSVs to `OUTPUT_DIR`.

In [3]:
analyse.run(MANIFEST_PATH, output_dir=OUTPUT_DIR, n_workers=4)

Stage 4: analysing 16/16 images (n_workers=4)
Stage 4 done: 16/16 images analysed.
  Z:\Bel\Marina_Side_Projects\CART analysis\To_Do\Outputs\all_counts.csv
  Z:\Bel\Marina_Side_Projects\CART analysis\To_Do\Outputs\all_distances.csv
  Z:\Bel\Marina_Side_Projects\CART analysis\To_Do\Outputs\all_colocalisation.csv


{'counts_csv': WindowsPath('Z:/Bel/Marina_Side_Projects/CART analysis/To_Do/Outputs/all_counts.csv'),
 'distances_csv': WindowsPath('Z:/Bel/Marina_Side_Projects/CART analysis/To_Do/Outputs/all_distances.csv'),
 'coloc_csv': WindowsPath('Z:/Bel/Marina_Side_Projects/CART analysis/To_Do/Outputs/all_colocalisation.csv')}

## Stage 5 — Figures

Generates two sets of outputs:

1. **Per-image QC PNG** — 2-row × 4-column figure (raw channels + mask overlays).
2. **Publication plots** — summary bar charts, distance histograms, and colocalisation line plots loaded from the Stage 4 CSVs.

Uses `stage3/<name>.tif` if present, else `stage2/<name>.tif`.

In [4]:
figures.run(MANIFEST_PATH, figures_dir=OUTPUT_DIR / "figures", n_workers=4)

Stage 5: generating QC figures for 16/16 images (n_workers=4)
  [OK  ] UTD_device1
  [OK  ] ARi_device1
  [OK  ] UTD_device2
  [OK  ] ARi_device2
  [OK  ] UTD_device3
  [OK  ] ARi_device3
  [OK  ] ARi_device4
  [OK  ] UTD_device4
  [OK  ] UTD_device1
  [OK  ] ARi_device1
  [OK  ] UTD_device2
  [OK  ] ARi_device2
  [OK  ] UTD_device3
  [OK  ] ARi_device3
  [OK  ] UTD_device4
  [OK  ] ARi_device4
Stage 5 QC figures: 16/16 rendered.
Publication plots written to Z:\Bel\Marina_Side_Projects\CART analysis\To_Do\Outputs\figures\publication
